# CSE570 Unit II
## Part 2 — Correlation, Outliers, and Feature Engineering

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_selection import SelectKBest, f_classif

In [ ]:
# This cell makes the notebook work both locally and in Google Colab.
from pathlib import Path
import pandas as pd

csv_name = "student_performance_eda.csv"
csv_path = Path(csv_name)

if not csv_path.exists():
    try:
        from google.colab import files
        print(f"Please upload {csv_name}")
        uploaded = files.upload()
        csv_path = Path(next(iter(uploaded.keys())))
    except ImportError:
        raise FileNotFoundError(
            f"{csv_name} was not found. Place it in the same folder as this notebook."
        )

df = pd.read_csv(csv_path)
print("Dataset loaded successfully.")
print("Shape:", df.shape)
df.head()

## Correlation matrix

In [ ]:
numeric_df = df.select_dtypes(include="number")
correlation_matrix = numeric_df.corr()
correlation_matrix.round(2)

## Correlation heatmap

In [ ]:
fig, ax = plt.subplots(figsize=(10,7))
image = ax.imshow(correlation_matrix, aspect="auto")
ax.set_xticks(range(len(correlation_matrix.columns)))
ax.set_yticks(range(len(correlation_matrix.columns)))
ax.set_xticklabels(correlation_matrix.columns, rotation=90)
ax.set_yticklabels(correlation_matrix.columns)
for i in range(len(correlation_matrix.columns)):
    for j in range(len(correlation_matrix.columns)):
        ax.text(j, i, f"{correlation_matrix.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
fig.colorbar(image)
plt.tight_layout()
plt.show()

## Highly correlated features

In [ ]:
absolute_corr = correlation_matrix.abs()
upper_triangle = absolute_corr.where(
    np.triu(np.ones(absolute_corr.shape), k=1).astype(bool)
)
high_corr = [col for col in upper_triangle.columns if any(upper_triangle[col] > 0.85)]
print("Highly correlated features:", high_corr)

## IQR outlier detection

In [ ]:
def find_iqr_outliers(dataframe, column):
    q1 = dataframe[column].quantile(0.25)
    q3 = dataframe[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    outliers = dataframe[(dataframe[column] < lower) | (dataframe[column] > upper)]
    return lower, upper, outliers

for col in ["Study_Hours_Per_Day", "Attendance_Percentage", "Previous_Score"]:
    lower, upper, outliers = find_iqr_outliers(df, col)
    print(f"\n{col}: lower={lower:.2f}, upper={upper:.2f}")
    print(outliers[["Student_ID", col]])

## Outlier capping

In [ ]:
df_capped = df.copy()
lower, upper, _ = find_iqr_outliers(df_capped, "Study_Hours_Per_Day")
df_capped["Study_Hours_Per_Day"] = df_capped["Study_Hours_Per_Day"].clip(lower, upper)
print("Original max:", df["Study_Hours_Per_Day"].max())
print("Capped max:", df_capped["Study_Hours_Per_Day"].max())

## Feature transformation

In [ ]:
engineered_df = df.copy()
engineered_df["Study_Hours_Log"] = np.log1p(engineered_df["Study_Hours_Per_Day"])
engineered_df["Assignments_Sqrt"] = np.sqrt(engineered_df["Assignments_Completed"])
engineered_df["Attendance_Level"] = pd.cut(
    engineered_df["Attendance_Percentage"],
    bins=[0,60,75,90,100],
    labels=["Low","Moderate","Good","Excellent"],
    include_lowest=True
)
engineered_df["Attendance_Study_Interaction"] = (
    engineered_df["Attendance_Percentage"] *
    engineered_df["Study_Hours_Per_Day"]
)
engineered_df.head()

## Basic feature selection

In [ ]:
X_numeric = df.select_dtypes(include="number").drop(columns=["Student_ID"])
X_numeric = X_numeric.fillna(X_numeric.median())
y = df["Placed"].map({"No":0, "Yes":1})

selector = SelectKBest(score_func=f_classif, k=4)
selector.fit(X_numeric, y)

pd.Series(selector.scores_, index=X_numeric.columns).sort_values(ascending=False)

## Student Practice

In [ ]:
# Write your solution here
